# WandB Hyperparameter Analysis
Pulls runs from a WandB project, finds best hyperparameters, averages over seeds, and reports best results per model × representation.

## 1. Configuration

In [23]:
# ── Install if needed ─────────────────────────────────────────────────────────
# !pip install wandb pandas tabulate rich

import wandb
import pandas as pd
import numpy as np
from IPython.display import display

# ── Project settings — edit these ─────────────────────────────────────────────
ENTITY        = "ai-re"          # WandB username or team name
PROJECT       = "mantra-full-sweep"         # WandB project name

# Column names as stored in WandB run config / summary
MODEL_KEY     = "model._target_"                # config key that identifies the model
REPR_KEY       = "dataset.transform.representation._target_"
EMBD_KEY      = "dataset.transform.embedding._target_"       # config key for the representation
SEED_KEY      = "seed"                 # config key for the random seed

# Metric to optimise (must exist in run.summary)
METRIC        = "val/bal_acc"         # e.g. "val/f1", "test/loss"
HIGHER_BETTER = True                   # set False for loss-type metrics

# Hyperparameter keys to consider when searching for the best config
# Everything in run.config that is NOT one of these three is treated as a HP
NON_HP_KEYS   = {MODEL_KEY, REPR_KEY, EMBD_KEY, SEED_KEY}

# Only include runs that finished successfully
FILTER_STATE  = "finished"             # None → include all states

# Extra summary metrics to show in the final table
EXTRA_METRICS = ["val/loss", "test/bal_acc"]   # set [] to skip

print("Configuration loaded.")

Configuration loaded.


## 2. Fetch runs from WandB

## 2. Helpers — flatten nested dicts

In [14]:
def flatten_dict(d: dict, parent_key: str = "", sep: str = ".") -> dict:
    """
    Recursively flatten a nested dictionary.

    Example
    -------
    >>> flatten_dict({"optimizer": {"lr": 1e-3, "betas": [0.9, 0.999]}, "epochs": 10})
    {'optimizer.lr': 0.001, 'optimizer.betas': [0.9, 0.999], 'epochs': 10}

    Only dicts are recursed into; lists / scalars are kept as-is.
    """
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items


def safe_get_nested(d: dict, dotted_key: str):
    """Retrieve a value from a nested dict using a dot-separated key path."""
    keys = dotted_key.split(".")
    for k in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
    return d

In [15]:
api = wandb.Api(timeout=60)

filters = {"state": FILTER_STATE} if FILTER_STATE else {}
runs = api.runs(f"{ENTITY}/{PROJECT}", filters=filters)

records = []
for run in runs:
    raw_cfg = dict(run.config)
    summary = dict(run.summary)

    # Skip runs missing the primary metric
    if METRIC not in summary:
        continue

    # ── Flatten entire config (handles arbitrarily deep nesting) ──────────────
    flat_cfg = flatten_dict(raw_cfg)

    # NON_HP_KEYS may use dot-notation if the identity fields are nested,
    # e.g. NON_HP_KEYS = {"experiment.model", "experiment.representation", "seed"}
    row = {
        "run_id":   run.id,
        "run_name": run.name,
        MODEL_KEY:  flat_cfg.get(MODEL_KEY),
        REPR_KEY:   flat_cfg.get(REPR_KEY),
        SEED_KEY:   flat_cfg.get(SEED_KEY),
        EMBD_KEY:   flat_cfg.get(EMBD_KEY),
        METRIC:     summary[METRIC],
    }

    # ── Collect hyperparameters (everything that is not model/repr/seed) ───────
    # The dot-separated flat key becomes the column name: "optimizer.lr", etc.
    for k, v in flat_cfg.items():
        if k not in NON_HP_KEYS:
            row[f"hp_{k}"] = v

    # ── Collect extra metrics ─────────────────────────────────────────────────
    flat_summary = flatten_dict(summary)
    for m in EXTRA_METRICS:
        row[m] = flat_summary.get(m, np.nan)

    records.append(row)

df = pd.DataFrame(records)

# ── Report any columns that still contain raw dicts (should be none) ──────────
dict_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, dict)).any()]
if dict_cols:
    print(f"WARNING: these columns still contain dicts — check NON_HP_KEYS or flatten_dict: {dict_cols}")
else:
    print("All columns fully flattened.")

print(f"Fetched {len(df)} valid runs  |  {len(df.columns)} columns")
df.head()['seed'].unique()

All columns fully flattened.
Fetched 120 valid runs  |  65 columns


array([41])

## 3. Identify hyperparameter columns

In [16]:

[c for c in df.columns if c.startswith("hp_")]

['hp_task.loss._target_',
 'hp_task.name',
 'hp_task.target',
 'hp_task.metrics.acc.average',
 'hp_task.metrics.acc._target_',
 'hp_task.metrics.bal_acc.average',
 'hp_task.metrics.bal_acc._target_',
 'hp_task.task_type',
 'hp_task.imbalanced',
 'hp_model.n_layers',
 'hp_model.sc_order',
 'hp_model.aggr_norm',
 'hp_model.conv_order',
 'hp_model.out_channels',
 'hp_model.hidden_channels',
 'hp_train',
 'hp_logger.name',
 'hp_logger.entity',
 'hp_logger.project',
 'hp_compile',
 'hp_dataset.root',
 'hp_dataset.reload',
 'hp_dataset.version',
 'hp_dataset.dimension',
 'hp_dataset.transform.embedding.perturb',
 'hp_dataset.transform.embedding.normalize',
 'hp_dataset.transform.embedding.propagate',
 'hp_dataset.transform.create_labels.source',
 'hp_dataset.transform.create_labels._target_',
 'hp_dataset.transform.select_features.dst',
 'hp_dataset.transform.select_features.src',
 'hp_dataset.transform.select_features._target_',
 'hp_dataset.transform.select_features.representation',
 'hp_d

In [24]:

VALID_HP_TYPES = ['dataset.transform', 'model', 'trainer']

HP_COLS = [c for c in df.columns if any([c.startswith("hp_"+s) for s in VALID_HP_TYPES])]
print(f"Detected {len(HP_COLS)} hyperparameter columns:")
print(HP_COLS)

# ── Make every HP column hashable (required for groupby) ─────────────────────
# Lists (e.g. Adam betas [0.9, 0.999]), sets, and numpy arrays are
# converted to their string representation so they can be used as group keys.
def make_hashable(val):
    if isinstance(val, (list, set)):
        return str(val)            # "[0.9, 0.999]"
    if isinstance(val, dict):
        return str(val)            # shouldn't appear after flatten, but just in case
    if isinstance(val, np.ndarray):
        return str(val.tolist())
    return val

converted = []
for col in HP_COLS:
    if df[col].apply(lambda x: isinstance(x, (list, set, dict, np.ndarray))).any():
        df[col] = df[col].apply(make_hashable)
        converted.append(col)

if converted:
    print(f"\nConverted {len(converted)} column(s) to str for hashability:")
    for c in converted:
        print(f"  {c}")
else:
    print("\nAll HP columns were already hashable.")

Detected 52 hyperparameter columns:
['hp_model.n_layers', 'hp_model.sc_order', 'hp_model.aggr_norm', 'hp_model.conv_order', 'hp_model.out_channels', 'hp_model.hidden_channels', 'hp_dataset.transform.embedding.perturb', 'hp_dataset.transform.embedding.normalize', 'hp_dataset.transform.embedding.propagate', 'hp_dataset.transform.create_labels.source', 'hp_dataset.transform.create_labels._target_', 'hp_dataset.transform.select_features.dst', 'hp_dataset.transform.select_features.src', 'hp_dataset.transform.select_features._target_', 'hp_dataset.transform.select_features.representation', 'hp_dataset.transform.representation_1.index', 'hp_dataset.transform.representation_1.signed', 'hp_dataset.transform.representation_2.index', 'hp_dataset.transform.representation_2.signed', 'hp_dataset.transform.representation_2._target_', 'hp_dataset.transform.representation_3.index', 'hp_dataset.transform.representation_3.signed', 'hp_dataset.transform.representation_3._target_', 'hp_dataset.transform.re

In [19]:
df.to_csv('csv/sccnn.csv')

## 4. Find best hyperparameter config per (model, representation)

Strategy:
1. Group by `(model, representation, hp_*)` — i.e. all HP axes.
2. Average the primary metric over seeds within each group.
3. Pick the HP config that maximises (or minimises) the seed-averaged metric.

In [34]:
df = pd.concat([pd.read_csv('csv/sccnn.csv'), pd.read_csv('csv/gnns.csv')], ignore_index=True)

In [36]:
df = df[~pd.isna(df[METRIC])]
#for MET in EXTRA_METRICS:
#    df = df[~pd.isna(df[MET])]

In [39]:
df[MODEL_KEY]

0       src.models.sccnn.SCCNN
1       src.models.sccnn.SCCNN
2       src.models.sccnn.SCCNN
3       src.models.sccnn.SCCNN
4       src.models.sccnn.SCCNN
                 ...          
1576       models.GCN.GCNHydra
1577       models.GCN.GCNHydra
1578       models.GCN.GCNHydra
1579       models.GCN.GCNHydra
1580       models.GCN.GCNHydra
Name: model._target_, Length: 1581, dtype: object

In [82]:
GROUP_KEYS = [MODEL_KEY, REPR_KEY, EMBD_KEY] + HP_COLS

# ── Aggregate over seeds ───────────────────────────────────────────────────────
agg_dict = {METRIC: ["mean", "std", "count"]}
for m in EXTRA_METRICS:
    if m in df.columns:
        agg_dict[m] = {"mean", "std"}

grouped = (
    df.groupby(GROUP_KEYS, dropna=False)
      .agg(agg_dict)
      .reset_index()
)

# Flatten MultiIndex columns
grouped.columns = [
    "_".join(filter(None, col))
    for col in grouped.columns.to_flat_index()
]

MEAN_COL  = f"{METRIC}_mean"
STD_COL   = f"{METRIC}_std"
COUNT_COL = f"{METRIC}_count"

print(f"Total HP configurations evaluated: {len(grouped)}")
grouped.head()

Total HP configurations evaluated: 539


,model._target_,dataset.transform.representation._target_,dataset.transform.embedding._target_,hp_model.n_layers,hp_model.sc_order,hp_model.aggr_norm,hp_model.conv_order,hp_model.out_channels,hp_model.hidden_channels,hp_dataset.transform.embedding.perturb,hp_dataset.transform.embedding.normalize,hp_dataset.transform.embedding.propagate,hp_dataset.transform.create_labels.source,hp_dataset.transform.create_labels._target_,hp_dataset.transform.select_features.dst,hp_dataset.transform.select_features.src,hp_dataset.transform.select_features._target_,hp_dataset.transform.select_features.representation,hp_dataset.transform.representation_1.index,hp_dataset.transform.representation_1.signed,hp_dataset.transform.representation_2.index,hp_dataset.transform.representation_2.signed,hp_dataset.transform.representation_2._target_,hp_dataset.transform.representation_3.index,hp_dataset.transform.representation_3.signed,hp_dataset.transform.representation_3._target_,hp_dataset.transform.representation_4.index,hp_dataset.transform.representation_4.signed,hp_dataset.transform.representation_4._target_,hp_dataset.transform_order,hp_trainer.device,hp_trainer.max_epoch,hp_trainer.batch_size,hp_trainer.weight_decay,hp_trainer.learning_rate,hp_trainer.early_stopping,hp_model.layers,hp_model.residuals,hp_model.batch_norm,hp_dataset.transform.embedding.attr_name,hp_dataset.transform.embedding.walk_length,hp_model.dropout,hp_model.num_heads,hp_model.max_degree,hp_model.max_distance,hp_model.spd_undirected,hp_model.num_hidden_layers,hp_model.num_node_features,hp_model.ffn_expansion_ratio,hp_model.use_spatial_encoding,hp_dataset.transform.graphormer_spd._target_,hp_dataset.transform.graphormer_spd.attr_name,hp_dataset.transform.graphormer_spd.undirected,hp_dataset.transform.graphormer_spd.max_distance,hp_dataset.transform.representation.feature_propagation,val/bal_acc_mean,val/bal_acc_std,val/bal_acc_count,val/loss_std,val/loss_mean,test/bal_acc_std,test/bal_acc_mean
0,models.GCN.GCNHydra,mantra.representations.DualGraph,mantra.transforms.MomentCurveEmbedding,8.0,NaN,NaN,NaN,9,64,False,False,False,name,mantra.transforms.CreateLabels,NaN,moment_curve_embedding,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['representation', 'embedding', 'select_featur...",cuda,300,128,0,0.001,40,NaN,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.755860,0.039098,5,0.164363,0.369603,0.038985,0.756154
1,models.GCN.GCNHydra,mantra.representations.DualGraph,mantra.transforms.MomentCurveEmbedding,8.0,NaN,NaN,NaN,9,64,False,False,False,name,mantra.transforms.CreateLabels,NaN,moment_curve_embedding,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['representation', 'embedding', 'select_featur...",cuda,300,256,0,0.001,40,NaN,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.743085,0.035521,5,0.052999,0.357231,0.035428,0.743316
2,models.GCN.GCNHydra,mantra.representations.DualGraph,mantra.transforms.MomentCurveEmbedding,8.0,NaN,NaN,NaN,9,64,False,False,False,name,mantra.transforms.CreateLabels,NaN,moment_curve_embedding,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['representation', 'embedding', 'select_featur...",cuda,300,256,0,0.010,40,NaN,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.711226,0.043873,5,1.209406,1.641138,0.044145,0.711324
3,models.GCN.GCNHydra,mantra.representations.DualGraph,mantra.transforms.MomentCurveEmbedding,8.0,NaN,NaN,NaN,9,128,False,False,False,name,mantra.transforms.CreateLabels,NaN,moment_curve_embedding,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['representation', 'embedding', 'select_featur...",cuda,300,128,0,0.001,40,NaN,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.858930,0.017019,5,0.013684,0.181337,0.016955,0.859040
4,mod

In [83]:
grouped[grouped[MODEL_KEY] == "src.models.sccnn.SCCNN"]
grouped.loc[grouped[MODEL_KEY]== "src.models.sccnn.SCCNN", 'dataset.transform.representation._target_'] = 'SC'

In [84]:
grouped

,model._target_,dataset.transform.representation._target_,dataset.transform.embedding._target_,hp_model.n_layers,hp_model.sc_order,hp_model.aggr_norm,hp_model.conv_order,hp_model.out_channels,hp_model.hidden_channels,hp_dataset.transform.embedding.perturb,hp_dataset.transform.embedding.normalize,hp_dataset.transform.embedding.propagate,hp_dataset.transform.create_labels.source,hp_dataset.transform.create_labels._target_,hp_dataset.transform.select_features.dst,hp_dataset.transform.select_features.src,hp_dataset.transform.select_features._target_,hp_dataset.transform.select_features.representation,hp_dataset.transform.representation_1.index,hp_dataset.transform.representation_1.signed,hp_dataset.transform.representation_2.index,hp_dataset.transform.representation_2.signed,hp_dataset.transform.representation_2._target_,hp_dataset.transform.representation_3.index,hp_dataset.transform.representation_3.signed,hp_dataset.transform.representation_3._target_,hp_dataset.transform.representation_4.index,hp_dataset.transform.representation_4.signed,hp_dataset.transform.representation_4._target_,hp_dataset.transform_order,hp_trainer.device,hp_trainer.max_epoch,hp_trainer.batch_size,hp_trainer.weight_decay,hp_trainer.learning_rate,hp_trainer.early_stopping,hp_model.layers,hp_model.residuals,hp_model.batch_norm,hp_dataset.transform.embedding.attr_name,hp_dataset.transform.embedding.walk_length,hp_model.dropout,hp_model.num_heads,hp_model.max_degree,hp_model.max_distance,hp_model.spd_undirected,hp_model.num_hidden_layers,hp_model.num_node_features,hp_model.ffn_expansion_ratio,hp_model.use_spatial_encoding,hp_dataset.transform.graphormer_spd._target_,hp_dataset.transform.graphormer_spd.attr_name,hp_dataset.transform.graphormer_spd.undirected,hp_dataset.transform.graphormer_spd.max_distance,hp_dataset.transform.representation.feature_propagation,val/bal_acc_mean,val/bal_acc_std,val/bal_acc_count,val/loss_std,val/loss_mean,test/bal_acc_std,test/bal_acc_mean
0,models.GCN.GCNHydra,mantra.representations.DualGraph,mantra.transforms.MomentCurveEmbedding,8.0,NaN,NaN,NaN,9,64,False,False,False,name,mantra.transforms.CreateLabels,NaN,moment_curve_embedding,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['representation', 'embedding', 'select_featur...",cuda,300,128,0,0.001,40,NaN,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.755860,0.039098,5,0.164363,0.369603,0.038985,0.756154
1,models.GCN.GCNHydra,mantra.representations.DualGraph,mantra.transforms.MomentCurveEmbedding,8.0,NaN,NaN,NaN,9,64,False,False,False,name,mantra.transforms.CreateLabels,NaN,moment_curve_embedding,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['representation', 'embedding', 'select_featur...",cuda,300,256,0,0.001,40,NaN,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.743085,0.035521,5,0.052999,0.357231,0.035428,0.743316
2,models.GCN.GCNHydra,mantra.representations.DualGraph,mantra.transforms.MomentCurveEmbedding,8.0,NaN,NaN,NaN,9,64,False,False,False,name,mantra.transforms.CreateLabels,NaN,moment_curve_embedding,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['representation', 'embedding', 'select_featur...",cuda,300,256,0,0.010,40,NaN,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.711226,0.043873,5,1.209406,1.641138,0.044145,0.711324
3,models.GCN.GCNHydra,mantra.representations.DualGraph,mantra.transforms.MomentCurveEmbedding,8.0,NaN,NaN,NaN,9,128,False,False,False,name,mantra.transforms.CreateLabels,NaN,moment_curve_embedding,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['representation', 'embedding', 'select_featur...",cuda,300,128,0,0.001,40,NaN,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.858930,0.017019,5,0.013684,0.181337,0.016955,0.859040
4,mod

In [85]:
# ── Select best HP config per (model, representation) ─────────────────────────
if HIGHER_BETTER:
    best_idx = grouped.groupby([MODEL_KEY, REPR_KEY, EMBD_KEY])[MEAN_COL].idxmax()
else:
    best_idx = grouped.groupby([MODEL_KEY, REPR_KEY, EMBD_KEY])[MEAN_COL].idxmin()

best_df = grouped.loc[best_idx].reset_index(drop=True)

print(f"Best configs found: {len(best_df)}")
best_df[[MODEL_KEY, REPR_KEY, EMBD_KEY, MEAN_COL, STD_COL, COUNT_COL]].head(50)

Best configs found: 19


,model._target_,dataset.transform.representation._target_,dataset.transform.embedding._target_,val/bal_acc_mean,val/bal_acc_std,val/bal_acc_count
0,models.GCN.GCNHydra,mantra.representations.DualGraph,mantra.transforms.MomentCurveEmbedding,0.910551,0.013832,5
1,models.GCN.GCNHydra,mantra.representations.DualGraph,torch_geometric.transforms.AddRandomWalkPE,0.923456,0.000782,2
2,models.GCN.GCNHydra,mantra.representations.HasseDiagram,mantra.transforms.MomentCurveEmbedding,0.939027,0.007739,3
3,models.GCN.GCNHydra,mantra.representations.HasseDiagram,torch_geometric.transforms.AddRandomWalkPE,0.924052,0.001636,2
4,models.GCN.GCNHydra,mantra.representations.OneSkeleton,mantra.transforms.MomentCurveEmbedding,0.873799,0.004035,2
5,models.GCN.GCNHydra,mantra.representations.OneSkeleton,torch_geometric.transforms.AddRandomWalkPE,0.930891,0.000743,2
6,models.Graphormer.Graphormer,mantra.representations.DualGraph,mantra.transforms.MomentCurveEmbedding,0.969211,NaN,1
7,models.Graphormer.Graphormer,mantra.representations.DualGraph,torch_geometric.transforms.AddRandomWalkPE,0.912469,0.006999,3
8,models.Graphormer.Graphormer,mantra.representations.HasseDiagram,mantra.transforms.MomentCurveEmbedding,0.924955,NaN,1
9,models.Graphormer.Graphormer,mantra.representations.HasseDiagram,torch_geometric.transforms.AddRandomWalkPE,0.938743,NaN,1


## 5. Summary table — best result per model × representation

In [44]:
# ── Build a clean display table ───────────────────────────────────────────────
extra_mean_cols = [f"{m}_mean" for m in EXTRA_METRICS if f"{m}_mean" in best_df.columns]
extra_std_cols = [f"{m}_std" for m in EXTRA_METRICS if f"{m}_std" in best_df.columns]

display_cols = (
    [MODEL_KEY, REPR_KEY]
    + [MEAN_COL, STD_COL, COUNT_COL]
    + extra_mean_cols
    + extra_std_cols
    + HP_COLS
)

summary_table = best_df[display_cols].copy()
summary_table = summary_table[summary_table[COUNT_COL] > 0]

# Pretty-print metric column names
rename = {
    MEAN_COL:  f"{METRIC} (mean)",
    STD_COL:   f"{METRIC} (std)",
    COUNT_COL: "n_seeds",
}
for m in EXTRA_METRICS:
    col = f"{m}_mean"
    if col in summary_table.columns:
        rename[col] = f"{m} (mean)"

summary_table.rename(columns=rename, inplace=True)


# Sort: first by model, then by descending primary metric
sort_col = f"{METRIC} (mean)"
summary_table.sort_values(
    [MODEL_KEY, sort_col],
    ascending=[True, not HIGHER_BETTER],
    inplace=True,
)

# Round floats for readability
float_cols = summary_table.select_dtypes(include="float").columns
summary_table[float_cols] = summary_table[float_cols].round(4)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
display(summary_table.reset_index(drop=True))

,model._target_,dataset.transform.representation._target_,val/bal_acc (mean),val/bal_acc (std),n_seeds,val/loss (mean),test/bal_acc (mean),val/loss_std,test/bal_acc_std,hp_model.n_layers,hp_model.sc_order,hp_model.aggr_norm,hp_model.conv_order,hp_model.out_channels,hp_model.hidden_channels,hp_dataset.transform.embedding.perturb,hp_dataset.transform.embedding.normalize,hp_dataset.transform.embedding.propagate,hp_dataset.transform.create_labels.source,hp_dataset.transform.create_labels._target_,hp_dataset.transform.select_features.dst,hp_dataset.transform.select_features.src,hp_dataset.transform.select_features._target_,hp_dataset.transform.select_features.representation,hp_dataset.transform.representation_1.index,hp_dataset.transform.representation_1.signed,hp_dataset.transform.representation_2.index,hp_dataset.transform.representation_2.signed,hp_dataset.transform.representation_2._target_,hp_dataset.transform.representation_3.index,hp_dataset.transform.representation_3.signed,hp_dataset.transform.representation_3._target_,hp_dataset.transform.representation_4.index,hp_dataset.transform.representation_4.signed,hp_dataset.transform.representation_4._target_,hp_dataset.transform_order,hp_trainer.device,hp_trainer.max_epoch,hp_trainer.batch_size,hp_trainer.weight_decay,hp_trainer.learning_rate,hp_trainer.early_stopping,hp_model.layers,hp_model.residuals,hp_model.batch_norm,hp_dataset.transform.embedding.attr_name,hp_dataset.transform.embedding.walk_length,hp_model.dropout,hp_model.num_heads,hp_model.max_degree,hp_model.max_distance,hp_model.spd_undirected,hp_model.num_hidden_layers,hp_model.num_node_features,hp_model.ffn_expansion_ratio,hp_model.use_spatial_encoding,hp_dataset.transform.graphormer_spd._target_,hp_dataset.transform.graphormer_spd.attr_name,hp_dataset.transform.graphormer_spd.undirected,hp_dataset.transform.graphormer_spd.max_distance,hp_dataset.transform.representation.feature_propagation
0,models.GCN.GCNHydra,mantra.representations.HasseDiagram,0.9390,0.0077,3,0.0858,0.9390,0.0205,0.0077,16.0,NaN,NaN,NaN,9,172,False,False,False,name,mantra.transforms.CreateLabels,NaN,moment_curve_embedding,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['representation', 'embedding', 'select_featur...",cuda,300,128,0,0.010,40,NaN,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,models.GCN.GCNHydra,mantra.representations.OneSkeleton,0.9309,0.0007,2,0.1603,NaN,0.0092,NaN,NaN,NaN,NaN,NaN,9,128,NaN,NaN,NaN,name,mantra.transforms.CreateLabels,x,random_walk_pe,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['representation', 'embedding', 'select_featur...",cuda,300,128,0,0.001,10,4.0,True,True,random_walk_pe,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,models.GCN.GCNHydra,mantra.representations.HasseDiagram,0.9241,0.0016,2,0.1752,NaN,0.0012,NaN,NaN,NaN,NaN,NaN,9,128,NaN,NaN,NaN,name,mantra.transforms.CreateLabels,x,random_walk_pe,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['representation', 'embedding', 'select_featur...",cuda,300,256,0,0.001,10,5.0,True,True,random_walk_pe,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,models.GCN.GCNHydra,mantra.representations.DualGraph,0.9235,0.0008,2,0.1470,NaN,0.0022,NaN,NaN,NaN,NaN,NaN,9,128,NaN,NaN,NaN,name,mantra.transforms.CreateLabels,x,random_walk_pe,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['representation', 'embedding', 'select_featur...",cuda,300,128,0,0.001,10,4.0,True,True,random_walk_pe,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,models.GCN.GCNHydra,mantra.representations.DualGraph,0.9106,0.0138,5,0.1023,0.9106,0.0395,0.0138,16.0,NaN,NaN,NaN,9,172,False,False,False,name,mantra.transforms.CreateLabels,NaN,moment_curve_embedding,mantra.transforms.select_features.SelectFeatures,graph,NaN,NaN,NaN

## 6. Pivot view — models as rows, representations as columns

In [10]:
pivot = summary_table.pivot(
    index=MODEL_KEY,
    columns=REPR_KEY,
    values=f"{METRIC} (mean)"
)

# Optional: highlight best value per row
def highlight_best(row):
    if HIGHER_BETTER:
        best_val = row.max()
    else:
        best_val = row.min()
    return ["font-weight: bold; background-color: #d4edda" if v == best_val else "" for v in row]

display(
    pivot.style
         .apply(highlight_best, axis=1)
         .format("{:.4f}", na_rep="—")
         .set_caption(f"Best {METRIC} (mean over seeds) — bold = best representation per model")
)

ValueError: Index contains duplicate entries, cannot reshape

## 7. Export to CSV and LaTeX

In [ ]:
# ── CSV ────────────────────────────────────────────────────────────────────────
summary_table.to_csv("best_results.csv", index=False)
print("Saved: best_results.csv")

# ── LaTeX (pivot only, cleaner for papers) ─────────────────────────────────────
latex_str = pivot.to_latex(
    float_format="{:.4f}".format,
    na_rep="—",
    caption=f"Best {METRIC} (mean over seeds) per model and representation.",
    label="tab:best_results",
    bold_rows=True,
)
with open("best_results.tex", "w") as f:
    f.write(latex_str)
print("Saved: best_results.tex")

Saved: best_results.csv
Saved: best_results.tex


## 8. (Optional) Per-seed detail for the best configs

In [ ]:
# Re-join raw runs with the chosen best HP configs to inspect seed-level variance

best_hp_df = best_df[[MODEL_KEY, REPR_KEY] + HP_COLS].copy()

seed_detail = df.merge(
    best_hp_df,
    on=[MODEL_KEY, REPR_KEY] + HP_COLS,
    how="inner",
)[["run_id", "run_name", MODEL_KEY, REPR_KEY, SEED_KEY, METRIC] + HP_COLS]

seed_detail.sort_values([MODEL_KEY, REPR_KEY, SEED_KEY], inplace=True)
display(seed_detail.reset_index(drop=True))

,run_id,run_name,model._target_,dataset.transform.representation._target_,seed,val/bal_acc,hp_task.loss._target_,hp_task.name,hp_task.target,hp_task.metrics.acc.average,hp_task.metrics.acc._target_,hp_task.metrics.bal_acc.average,hp_task.metrics.bal_acc._target_,hp_task.task_type,hp_task.imbalanced,hp_model.layers,hp_model.residuals,hp_model.batch_norm,hp_model.out_channels,hp_model.hidden_channels,hp_train,hp_logger.name,hp_logger.project,hp_dataset.root,hp_dataset.reload,hp_dataset.version,hp_dataset.dimension,hp_dataset.transform.embedding.attr_name,hp_dataset.transform.embedding.walk_length,hp_dataset.transform.create_labels.source,hp_dataset.transform.create_labels._target_,hp_dataset.transform.select_features.dst,hp_dataset.transform.select_features.src,hp_dataset.transform.select_features._target_,hp_dataset.transform.select_features.representation,hp_dataset.use_stratified,hp_dataset.transform_order,hp_dataset.train_parameters.split.val,hp_dataset.train_parameters.split.test,hp_dataset.train_parameters.split.train,hp_trainer.device,hp_trainer.max_epoch,hp_trainer.batch_size,hp_trainer.weight_decay,hp_trainer.learning_rate,hp_trainer.early_stopping,hp_dataset.transform.embedding.perturb,hp_dataset.transform.embedding.normalize,hp_logger.entity,hp_dataset.num_features,hp_model.dropout,hp_model.num_heads,hp_model.max_degree,hp_model.max_distance,hp_model.spd_undirected,hp_model.num_hidden_layers,hp_model.num_node_features,hp_model.ffn_expansion_ratio,hp_model.use_spatial_encoding,hp_dataset.transform.graphormer_spd._target_,hp_dataset.transform.graphormer_spd.attr_name,hp_dataset.transform.graphormer_spd.undirected,hp_dataset.transform.graphormer_spd.max_distance,hp_model.n_layers,hp_compile,hp_checkpoint_path,hp_dataset.transform.embedding.propagate,hp_dataset.transform.representation.feature_propagation,hp_dataset.local_path
0,k0i65yt0,icy-sponge-1309,models.GCN.GCNHydra,mantra.representations.DualGraph,42,0.928927,torch.nn.CrossEntropyLoss,Name,name,macro,torchmetrics.Accuracy,weighted,torchmetrics.Accuracy,classification,False,NaN,True,True,9,172,True,WanDB,mantra-sweep-full,/tmp/mantra_data_0_42_moment_curve_embedding_d...,False,latest,2,NaN,NaN,name,mantra.transforms.CreateLabels,None,moment_curve_embedding,mantra.transforms.select_features.SelectFeatures,graph,True,"['representation', 'embedding', 'select_featur...",0.2,0.2,0.6,cuda,300,128,0,0.001,40,False,False,ai-re,auto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,True,/tmp/mantra_checkpoints_0_moment_curve_embeddi...,False,NaN,NaN
1,7wgyupez,ancient-grass-1240,models.GCN.GCNHydra,mantra.representations.HasseDiagram,41,0.945783,torch.nn.CrossEntropyLoss,Name,name,macro,torchmetrics.Accuracy,weighted,torchmetrics.Accuracy,classification,False,NaN,True,True,9,172,True,WanDB,mantra-sweep-full,/tmp/mantra_data_1_41_moment_curve_embedding_h...,False,latest,2,NaN,NaN,name,mantra.transforms.CreateLabels,None,moment_curve_embedding,mantra.transforms.select_features.SelectFeatures,graph,True,"['representation', 'embedding', 'select_featur...",0.2,0.2,0.6,cuda,300,128,0,0.010,40,False,False,ai-re,auto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,True,/tmp/mantra_checkpoints_1_moment_curve_embeddi...,False,NaN,NaN
2,4r3kyh7d,toasty-sea-1128,models.GCN.GCNHydra,mantra.representations.OneSkeleton,44,0.934593,torch.nn.CrossEntropyLoss,Name,name,macro,torchmetrics.Accuracy,weighted,torchmetrics.Accuracy,classification,False,NaN,True,True,9,172,True,WanDB,mantra-sweep-full,/tmp/mantra_data_1_44_random_walk_pe_one_skeleton,False,latest,2,random_walk_pe,8.0,name,mantra.transforms.CreateLabels,None,random_walk_pe,mantra.transforms.select_features.SelectFeatures,graph,True,"['representation', 'embedding', 'select_featur...",0.2,0.2,0.6,cuda,300,128,0,0.001,40,NaN,NaN,ai-re,auto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,True,/tmp/mantra_checkpoints_1_random_walk_pe_one_s...,NaN,NaN,NaN
3,ji121zbl,exalted-elevator-651,models.Graphormer.Graphorme